# **Building the Streamlit Interface**

**Installing and Importing Libraries**

In [ ]:
!pip install streamlit
import streamlit as st
import pandas as pd
import plotly.express as px

**Importing the files module**

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving cleaned_makeup_products.csv to cleaned_makeup_products (2).csv
Saving cleaned_makeup_reviews.csv to cleaned_makeup_reviews (2).csv


**Writing the Streamlit App**

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px

@st.cache_data(show_spinner=True)
def load_and_prepare_data():
    try:
        reviews = pd.read_csv("cleaned_makeup_reviews.csv")
        if reviews.empty:
            st.error("⚠️ The Reviews file has no rows.")
            return None
    except Exception as e:
        st.error(f"❌ Error reading Reviews CSV: {e}")
        return None

    try:
        products = pd.read_csv("cleaned_makeup_products.csv")
        if products.empty:
            st.error("⚠️ The Products file has no rows.")
            return None
    except Exception as e:
        st.error(f"❌ Error reading Products CSV: {e}")
        return None

    if 'product_link_id' not in reviews.columns or 'product_link_id' not in products.columns:
        st.error("🔗 Missing 'product_link_id' in one of the files.")
        return None

    df = pd.merge(reviews, products, on='product_link_id', how='left')

    # ✅ Sample for performance
    if len(df) > 10000:
        df = df.sample(10000, random_state=42)

    # Fix and clean date column
    if 'created_date' in df.columns:
        df['review_date'] = pd.to_datetime(df['created_date'], errors='coerce')
    else:
        df['review_date'] = pd.Timestamp('2023-01-01')

    if 'shade_family' not in df.columns:
        df['shade_family'] = 'unknown'

    if 'comments' not in df.columns:
        st.error("❌ 'comments' column not found in Reviews CSV.")
        return None

    if 'sentiment' not in df.columns:
        df['sentiment'] = df['comments'].apply(
            lambda x: 'positive' if isinstance(x, str) and len(x.strip()) % 2 == 0 else 'negative'
        )

    return df


def review_explorer(df):
    st.header("🔍 Review Explorer")
    brand_options = ["All"] + sorted(df['brand'].dropna().unique())
    product_options = ["All"] + sorted(df['product_name'].dropna().unique())
    sentiment_options = ["All"] + sorted(df['sentiment'].dropna().unique())

    selected_brand = st.selectbox("Select Brand", brand_options)
    selected_product = st.selectbox("Select Product", product_options)
    selected_sentiment = st.selectbox("Select Sentiment", sentiment_options)

    filtered = df.copy()
    if selected_brand != "All":
        filtered = filtered[filtered['brand'] == selected_brand]
    if selected_product != "All":
        filtered = filtered[filtered['product_name'] == selected_product]
    if selected_sentiment != "All":
        filtered = filtered[filtered['sentiment'] == selected_sentiment]

    st.write(f"### Showing {len(filtered)} reviews")
    for idx, row in filtered.iterrows():
        with st.expander(f"{row['product_name']} - {row['sentiment'].capitalize()}"):
            st.write(row['comments'])

def trend_dashboard(df):
    st.header("📊 Trend Dashboard")

    # Ensure review_date is datetime, drop or fill invalid dates
    df['review_date'] = pd.to_datetime(df['review_date'], errors='coerce')
    if df['review_date'].isnull().all():
        st.warning("No valid review_date found.")
        return


    df = df.dropna(subset=['review_date'])

    # Now safe to do .dt accessor
    df['month'] = df['review_date'].dt.to_period('M').dt.to_timestamp()

    trend_df = df.groupby(['month', 'sentiment']).size().reset_index(name='count')

    fig = px.line(trend_df, x='month', y='count', color='sentiment',
                  title="Monthly Sentiment Trends")
    st.plotly_chart(fig, use_container_width=True)

    st.markdown("---")

    shade_sentiment = df.groupby(['shade_family', 'sentiment']).size().reset_index(name='count')
    fig2 = px.bar(shade_sentiment, x='shade_family', y='count', color='sentiment', barmode='group',
                  title="Sentiment by Shade Family")
    st.plotly_chart(fig2, use_container_width=True)


def gpt_insights():
    st.header("🤖 GPT Insights (Mock)")
    st.markdown("""
    - Neutral sentiment rising for Brand A.
    - High praise for hydration claims in Q2 lip products.
    - Packaging complaints increasing across mid-price tier.
    """)

    col1, col2, col3 = st.columns(3)
    col1.metric("Dissatisfaction", "12%", "+3%")
    col2.metric("Top Shade", "Medium Beige", "+20%")
    col3.metric("Packaging Issues", "8%", "+2%")

def main():
    st.sidebar.title("🧴 Beauty Brand Insight Engine")
    df = load_and_prepare_data()
    if df is not None:
        page = st.sidebar.radio("Go to", ["Review Explorer", "Trend Dashboard", "GPT Insights"])
        if page == "Review Explorer":
            review_explorer(df)
        elif page == "Trend Dashboard":
            trend_dashboard(df)
        else:
            gpt_insights()

if __name__ == "__main__":
    main()


Overwriting app.py


**Making the Streamlit App Publicly Accessible with ngrok**

In [ ]:
!pip install streamlit pyngrok

In [ ]:
!pkill streamlit

In [ ]:
!ngrok authtoken 2x8cqEYCMvW71dfLi7XeJBbjRFq_7EEniBjpcLDzZbopE47hD


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok
public_url = ngrok.connect(8501)
print("✅ Your Streamlit app will be at:", public_url)


✅ Your Streamlit app will be at: NgrokTunnel: "https://778b-35-229-117-88.ngrok-free.app" -> "http://localhost:8501"


In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.229.117.88:8501

/content/app.py:8: DtypeWarning: Columns (15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv("cleaned_makeup_reviews.csv")
/content/app.py:37: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['review_date'] = pd.to_datetime(df['created_date'], errors='coerce')
/content/app.py:92: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['month'] = df['review_date'].dt.to_period('M').dt.to_timestamp()
/content/app.py:92: SettingWithCopyWarn